In [61]:
from typing import TypedDict, Literal

class Ingredient(TypedDict):
    name: str
    amount: float
    unit: str
    flavour: Literal[
        'acidity', 'salt', 'spice', 'bitter', 
        'creamy', 'sweet', 'none', 'aromatic'
        ]
    type: Literal[
        'starches', 'sugars', 'salts', 'fats', 
        'seeds_nuts_dried_fruits', 'binders', 
        'produce', 'spices', 'sugars', 'liquids'
        ]
    ingredients: list[Ingredient]
    cons: Literal["SOLID", "LIQUID"]


class Recipe(TypedDict):
    name: str
    servings: int
    ingredients: list[Ingredient]
    source: str

In [62]:
import json
from typing import Dict, List, Any, Literal
from load_json import load_json_file
import pandas as pd
initial_dataset = []
paths:list[str] = [
    '../data/western_cookies.json',
    '../data/indonesian_baked_goods.json',
    # '../data/indonesian_drinks.json'
]

for path in paths:
    data:list[Dict[unknown, Unknown]] = load_json_file(path)
    initial_dataset.extend(data)

def find_and_replace_recipe_properties(
    inspiring_set: list[Recipe],
    replacement_data: pd.DataFrame,
    target_properties: list[str] | None = None,
    reference_column: str = "ingredient",
) -> list[Recipe]:
    properties = target_properties or ["flavour", "type", "is_indonesian"]

    if reference_column not in replacement_data.columns:
        raise KeyError(
            f"Column {reference_column!r} not found in metadata. "
            f"Available columns: {list(replacement_data.columns)}"
        )

    for recipe in inspiring_set:
        for ingredient in recipe["ingredients"]:
            ingredient_name = ingredient.get("name")
            if not ingredient_name:
                continue

            matching_rows = replacement_data[
                replacement_data[reference_column] == ingredient_name
            ]

            if matching_rows.empty:
                continue

            row = matching_rows.iloc[0]
            for prop in properties:
                if prop in matching_rows.columns and pd.notna(row[prop]):
                    value = row[prop]
                    ingredient[prop] = value.item() if hasattr(value, 'item') else value

    return inspiring_set

def norm_data(recipes:list[Recipe]) -> list[Recipe]:
    for recipe in recipes:
        for i in recipe["ingredients"]:
            i["source"]=recipe["name"]
            if i["unit"] in ["tsp", "tsps"]:
                i["unit"]="g" if i["cons"]=="SOLID" else "ml"
                i["amount"]=i["amount"]*5.0

            elif i["unit"] in ["Tbsp", "Tbsps"]:
                i["unit"]="g" if i["cons"]=="SOLID" else "ml"
                i["amount"]=i["amount"]*15.0
            elif i["unit"]=="pkt":
                i["unit"]="g"
                i["amount"]=i["amount"]*16
            elif i["unit"]=="pinch":
                i["unit"]="g" if i["cons"]=="SOLID" else "ml"
                i["amount"]=i["amount"]*0.5
            elif i["unit"] in ["stick", "sticks"]:
                i["unit"]="g"
                i["amount"]=i["amount"]*113
            elif i["name"] in ["eggs", "egg", "egg yolk", "egg yolks"]:
                if i["unit"]=="large":
                    i["amount"] = i["amount"] * 55.0
                else:
                    i["amount"] = i["amount"] * 50.0
                i["unit"]="g"
            elif i["unit"] in ["cup", "cups"]:
                if i["cons"] == "LIQUID":
                        i["amount"] = i["amount"] * 240.0
                        i["unit"] = "ml"
                else:
                    i["amount"] = i["amount"] * 130.0
                    i["unit"] = "g"
    return recipes

print(initial_dataset)

classifiers = pd.read_csv("../data/ingredient_classification.csv")

data = norm_data(initial_dataset)
data = find_and_replace_recipe_properties(data, classifiers)
file = open("inspiring_cookies.json", 'w')
file.write(json.dumps(data))
file.close()


[{'name': 'Sugar Cookies', 'family': 'cookie', 'servings': 16, 'ingredients': [{'name': 'flour', 'amount': 500.0, 'unit': 'g', 'type': 'Baking', 'cons': 'SOLID'}, {'name': 'salt', 'amount': 0.5, 'unit': 'tsps', 'type': 'Spices and Seasonings', 'cons': 'SOLID'}, {'name': 'baking powder', 'amount': 1.0, 'unit': 'tsp', 'type': 'Baking', 'cons': 'SOLID'}, {'name': 'butter', 'amount': 227.0, 'unit': 'g', 'type': 'Milk, Eggs, Other Dairy', 'cons': 'SOLID'}, {'name': 'sugar', 'amount': 400.0, 'unit': 'g', 'type': 'Baking', 'cons': 'SOLID'}, {'name': 'eggs', 'amount': 2.0, 'unit': 'large', 'type': 'Milk, Eggs, Other Dairy', 'cons': 'SOLID'}, {'name': 'lemon juice', 'amount': 2.0, 'unit': 'tsps', 'type': 'Baking', 'cons': 'LIQUID'}, {'name': 'lemon zest', 'amount': 2.0, 'unit': '', 'type': 'Produce', 'cons': 'SOLID'}]}, {'name': 'Muesli Cookies', 'family': 'cookie', 'servings': 8, 'ingredients': [{'name': 'flour', 'amount': 180.0, 'unit': 'g', 'type': 'Baking', 'cons': 'SOLID'}, {'name': 'muesl

In [63]:

inspiring_set:list[Dict[Unknown, Unknown]] = []

paths:list[str] = [
    # '../data/western_cookies.json',
    # '../data/indonesian_baked_goods.json',
    # '../data/indonesian_drinks.json'
    '../data/inspiring_set.json'
]

for path in paths:
    data:list[Dict[Unknown, Unknown]] = load_json_file(path)
    inspiring_set.extend(data)

print(str(inspiring_set))

[{'name': 'Sugar Cookies', 'family': 'cookie', 'servings': 16, 'ingredients': [{'name': 'flour', 'amount': 500.0, 'unit': 'g', 'type': 'starches', 'cons': 'SOLID', 'source': 'Sugar Cookies', 'flavour': 'none', 'is_indonesian': False}, {'name': 'salt', 'amount': 2.5, 'unit': 'g', 'type': 'salts', 'cons': 'SOLID', 'source': 'Sugar Cookies', 'flavour': 'salt', 'is_indonesian': False}, {'name': 'baking powder', 'amount': 5.0, 'unit': 'g', 'type': 'leaving_agents', 'cons': 'SOLID', 'source': 'Sugar Cookies', 'flavour': 'none', 'is_indonesian': False}, {'name': 'butter', 'amount': 227.0, 'unit': 'g', 'type': 'fats', 'cons': 'SOLID', 'source': 'Sugar Cookies', 'flavour': 'creamy', 'is_indonesian': False}, {'name': 'sugar', 'amount': 400.0, 'unit': 'g', 'type': 'sugars', 'cons': 'SOLID', 'source': 'Sugar Cookies', 'flavour': 'sweet', 'is_indonesian': False}, {'name': 'eggs', 'amount': 110.0, 'unit': 'g', 'type': 'binders', 'cons': 'SOLID', 'source': 'Sugar Cookies', 'flavour': 'none', 'is_indo

In [64]:
def get_unique_values(key:str) -> set:
    return {
        ingredient[key]
        for recipe in inspiring_set
        for ingredient in recipe["ingredients"]
        if key in ingredient
    }

unique_flavours = get_unique_values("flavour")
unique_types = get_unique_values("type")
unique_ingredients = get_unique_values("name")
unique_families = {recipe["family"] for recipe in inspiring_set}

print(unique_flavours)
print(unique_types)
print(unique_ingredients)
print(unique_families)

{'bitter', 'salt', 'spice', 'acidity', 'aromatic', 'none', 'sweet', 'creamy'}
{'sugars', 'fats', 'seeds_nuts_dried_fruits', 'spices', 'starches', 'produce', 'liquids', 'salts', 'binders', 'leaving_agents'}
{'ginger root', 'butter-flavored crisco', 'amaretto', 'basil seed', 'vanilla bean paste', 'old fashioned rolled oats', 'rolled oats', 'brown sugar and cinnamon oatmeal', 'butterscotch morsels', 'macadamia nuts', 'oats oats', 'pastry flour', 'orange juice', 'coconut', 'duncan hines classic decadent cake mix', 'young coconut flesh', 'coconut oil', 'condensed milk', 'almonds', 'pink lemonade kool-aid', 'gound almonds', 'unbleached flour', 'lemons', 'egg white', 'pink food coloring', 'confectioners sugar', 'self-raising flour', 'baking powder', 'pastured eggs', 'raspberry jam', 'lemon kool-aid', 'espresso powder', 'crisco', 'soy milk', 'greek yogurt', 'water', 'carrots', 'raisins', 'natural cocoa powder', 'xanthan gum', 'pepper', 'heath candy bars', 'pistachio oil', 'walnuts and pecans',

In [65]:
%pip install scipy
from typing import TypedDict
import pprint
import numpy as np
from scipy import stats
recipe_flavour_ratios = []

# def get_ingredient_ratios(recipe: Recipe, key: str) -> float:
#     total_recipe_amount = sum(ing["amount"] for ing in recipe["ingredients"])
#     amounts_per_category = {}
#     for ingredient in recipe["ingredients"]:
#         category = ingredient[key]
#         amounts_per_category[category] =+ ingredient["amount"]

#     for category, amount in amounts_per_category.items():
#         amounts_per_category[category] = amount / total_recipe_amount if total_recipe_amount > 0 else 0

#     return amounts_per_category

def get_ingredient_ratios(recipe: Dict[str, Any], key: str) -> Dict[str, float]:
    total_recipe_amount = sum(
        ingredient["amount"] for ingredient in recipe["ingredients"]
    )

    amounts_per_category: Dict[str, float] = {}

    for ingredient in recipe["ingredients"]:
        if key not in ingredient:
            continue
        category = str(ingredient[key])
        amounts_per_category[category] = (
            amounts_per_category.get(category, 0) + ingredient["amount"]
        )

    return {
        category: amount / total_recipe_amount if total_recipe_amount > 0 else 0
        for category, amount in amounts_per_category.items()
    }

class RatioResult(TypedDict):
    name: str
    ratios: dict[str, float]


class CategoryStatistics(TypedDict):
    mean: float
    min: float
    max: float


def calculate_category_statistics(
    recipes: list[dict[str, Any]],
    category_key: str,
) -> tuple[list[RatioResult], dict[str, CategoryStatistics]]:
    categories: set[str] = {
        str(ingredient[category_key])
        for recipe in recipes
        for ingredient in recipe["ingredients"]
        if category_key in ingredient
    }

    recipe_ratios: list[RatioResult] = []

    for recipe in recipes:
        ratios = get_ingredient_ratios(recipe, category_key)

        complete_ratios = {
            category: ratios.get(category, 0.0)
            for category in categories
        }

        recipe_ratios.append({
            "name": str(recipe["name"]),
            "ratios": complete_ratios,
        })

    statistics: list[tuple[str, float]] = [
        (category, sum(item["ratios"][category] for item in recipe_ratios) / len(recipe_ratios))
        for category in categories
    ]
    return recipe_ratios, statistics


recipe_flavour_ratios, flavour_statistics = calculate_category_statistics(
    inspiring_set,
    "flavour",
)

recipe_type_ratios, type_statistics = calculate_category_statistics(
    inspiring_set,
    "type",
)

# print("Flavour Statistics:", pprint.pformat(flavour_statistics))
print("Type Statistics:", pprint.pformat(type_statistics))
print("Flavour Statistics:", pprint.pformat(flavour_statistics))

Note: you may need to restart the kernel to use updated packages.
Type Statistics: [('sugars', 0.2932170227765012),
 ('fats', 0.1637691283588514),
 ('seeds_nuts_dried_fruits', 0.07440012600154759),
 ('spices', 0.012830196937252392),
 ('starches', 0.23673292553476527),
 ('produce', 0.04144882367505669),
 ('liquids', 0.10988447132245965),
 ('salts', 0.0023546964749862035),
 ('binders', 0.061411696648772864),
 ('leaving_agents', 0.003950912269806747)]
Flavour Statistics: [('bitter', 0.01743159535555998),
 ('salt', 0.0030019334438285555),
 ('spice', 0.0037138234291012166),
 ('acidity', 0.012523512271884623),
 ('aromatic', 0.011957355039979757),
 ('none', 0.4078520474875473),
 ('sweet', 0.3515561136931783),
 ('creamy', 0.18796792403248808)]
